# Quick Experimental Prompt Test - Notebook Version

Interactive notebook for testing experimental prompts with vision-language models.
Displays the prompt, image, and rendered markdown output.

In [ ]:
#!/usr/bin/env python3
"""
Quick Prompt Test Notebook - Interactive experimental prompt testing

Test prompts on vision-language models with visual output display.
Edit EXPERIMENTAL_PROMPT and run cells to test.
"""

import sys
from pathlib import Path
from PIL import Image
from IPython.display import display, Markdown, HTML
import time

# Add project root to path (parent directory since we're in notebooks/)
project_root = Path('..').absolute()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"📂 Project root: {project_root}")
print("✅ Environment configured")

In [ ]:
# Import the experimental prompt tester
try:
    from experimental_prompt_tester import ExperimentalPromptTester
    print("✅ ExperimentalPromptTester imported successfully")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("💡 Make sure experimental_prompt_tester.py exists in the parent directory (project root)")

## Configuration

Edit the cells below to configure your experiment:

In [ ]:
# ============================================================================
# EDIT THIS SECTION FOR YOUR EXPERIMENT
# ============================================================================

# Choose model: "llama" or "internvl3"
MODEL = "llama"

# Your experimental prompt (edit this!)
EXPERIMENTAL_PROMPT = """
You are a markdown converter. Look at this document image and recreate it as markdown:

1. Identify the document structure (headers, paragraphs, lists)
2. Preserve formatting (bold, italic, emphasis)
3. Convert tables to markdown table format
4. Output only the markdown, no explanations

Begin:"""

# Test image (or None to use first available)
TEST_IMAGE = "../evaluation_data/synthetic_invoice_001.png"

# Compare with baseline? (True/False)
COMPARE_WITH_BASELINE = False

# Maximum tokens for generation (increase for longer outputs)
MAX_TOKENS = 2048

print(f"🚀 Configuration:")
print(f"   Model: {MODEL}")
print(f"   Max tokens: {MAX_TOKENS}")
print(f"   Image: {TEST_IMAGE or 'auto-select'}")
print(f"   Compare baseline: {COMPARE_WITH_BASELINE}")

## Display Prompt

In [ ]:
# Display the experimental prompt
display(HTML("<h3>📝 Experimental Prompt:</h3>"))
display(Markdown(f"```\n{EXPERIMENTAL_PROMPT}\n```"))
print(f"\n📊 Prompt statistics:")
print(f"   Length: {len(EXPERIMENTAL_PROMPT)} characters")
print(f"   Lines: {len(EXPERIMENTAL_PROMPT.splitlines())}")
print(f"   Words: {len(EXPERIMENTAL_PROMPT.split())}")

## Load and Display Image

In [ ]:
# Auto-select image if none specified
image_path = TEST_IMAGE
if not image_path:
    try:
        from common.extraction_parser import discover_images
        images = discover_images("../evaluation_data/")
        if images:
            image_path = str(images[0])
            print(f"🎯 Auto-selected: {Path(image_path).name}")
        else:
            print("❌ No test images found in ../evaluation_data/")
    except Exception as e:
        print(f"❌ Error finding images: {e}")

# Load and display the image
if image_path and Path(image_path).exists():
    img = Image.open(image_path)
    
    display(HTML(f"<h3>🖼️ Test Image: {Path(image_path).name}</h3>"))
    
    # Resize for display if too large
    max_display_width = 800
    if img.width > max_display_width:
        ratio = max_display_width / img.width
        new_height = int(img.height * ratio)
        img_display = img.resize((max_display_width, new_height), Image.Resampling.LANCZOS)
        display(img_display)
        print(f"📐 Original size: {img.width}x{img.height}")
        print(f"📐 Display size: {img_display.width}x{img_display.height}")
    else:
        display(img)
        print(f"📐 Image size: {img.width}x{img.height}")
else:
    print(f"❌ Image not found: {image_path}")

## Run Model Processing

In [ ]:
# Initialize the tester
print(f"🚀 Initializing {MODEL} for experimental prompt testing...")
print(f"🎯 Max tokens: {MAX_TOKENS}")

try:
    tester = ExperimentalPromptTester(MODEL, debug=False, max_tokens=MAX_TOKENS)
    print(f"✅ {MODEL} initialized successfully")
except Exception as e:
    print(f"❌ Error initializing model: {e}")
    print("💡 Make sure you're running this on a machine with GPU and model access")
    raise

In [ ]:
# Run the experimental prompt
display(HTML("<h3>🔬 Processing with Model...</h3>"))

start_time = time.time()
result = None

try:
    if COMPARE_WITH_BASELINE:
        print("⚖️  Running comparison with baseline...\n")
        result = tester.compare_with_baseline(EXPERIMENTAL_PROMPT, image_path)
        
        # Extract responses
        baseline_response = result["baseline"]["response"]
        experimental_response = result["experimental"]["response"]
        
        print("\n📊 COMPARISON RESULTS:")
        print(f"   Baseline length: {len(baseline_response)} chars")
        print(f"   Experimental length: {len(experimental_response)} chars")
        print(f"   Time difference: {result['time_difference']:+.2f}s")
        
        # Store the experimental response for display
        model_response = experimental_response
        
    else:
        print("🧪 Running experimental prompt only...\n")
        result = tester.test_prompt(EXPERIMENTAL_PROMPT, image_path)
        model_response = result['response']
        
    processing_time = time.time() - start_time
    
    print(f"\n✅ Processing completed in {processing_time:.2f} seconds")
    print(f"📏 Response length: {len(model_response)} characters")
    
except Exception as e:
    print(f"❌ Error during processing: {e}")
    import traceback
    traceback.print_exc()
    model_response = None

## Display Results

In [ ]:
# Display raw model output
if model_response:
    display(HTML("<h3>🤖 Raw Model Output:</h3>"))
    display(HTML(f'<div style="background-color: #f0f0f0; padding: 10px; border-radius: 5px; font-family: monospace; white-space: pre-wrap; max-height: 400px; overflow-y: auto;">{model_response}</div>'))
else:
    print("❌ No model response to display")

In [ ]:
# Display rendered markdown (if applicable)
if model_response and "markdown" in EXPERIMENTAL_PROMPT.lower():
    display(HTML("<h3>📄 Rendered Markdown Output:</h3>"))
    display(HTML('<div style="border: 2px solid #ccc; padding: 20px; border-radius: 5px; background-color: white;">'))
    display(Markdown(model_response))
    display(HTML('</div>'))
else:
    print("💡 Tip: Include 'markdown' in your prompt to see rendered output")

## Comparison Results (if enabled)

In [ ]:
# Display baseline comparison if enabled
if COMPARE_WITH_BASELINE and result:
    display(HTML("<h3>⚖️ Baseline vs Experimental Comparison:</h3>"))
    
    baseline_response = result["baseline"]["response"]
    experimental_response = result["experimental"]["response"]
    
    # Create side-by-side comparison
    html_comparison = f"""
    <div style="display: flex; gap: 20px;">
        <div style="flex: 1;">
            <h4>📊 Baseline (Schema-Driven)</h4>
            <div style="background-color: #f8f8f8; padding: 10px; border-radius: 5px; font-family: monospace; font-size: 12px; white-space: pre-wrap; max-height: 400px; overflow-y: auto;">{baseline_response[:1000]}{'...' if len(baseline_response) > 1000 else ''}</div>
            <p>Length: {len(baseline_response)} chars | Time: {result['baseline']['processing_time']:.2f}s</p>
        </div>
        <div style="flex: 1;">
            <h4>🧪 Experimental</h4>
            <div style="background-color: #f0f8ff; padding: 10px; border-radius: 5px; font-family: monospace; font-size: 12px; white-space: pre-wrap; max-height: 400px; overflow-y: auto;">{experimental_response[:1000]}{'...' if len(experimental_response) > 1000 else ''}</div>
            <p>Length: {len(experimental_response)} chars | Time: {result['experimental']['processing_time']:.2f}s</p>
        </div>
    </div>
    """
    display(HTML(html_comparison))
    
    # Summary statistics
    print("\n📈 Summary:")
    print(f"   Time difference: {result['time_difference']:+.2f}s ({'+faster' if result['time_difference'] < 0 else '+slower'})")
    print(f"   Length difference: {len(experimental_response) - len(baseline_response):+d} chars")

## Quick Iteration Zone

Use the cells below for rapid prompt iteration:

In [ ]:
# Quick test with a new prompt (edit and run)
QUICK_PROMPT = """
Extract all text from this image and format it as a markdown document.
Maintain the original structure and formatting.
"""

print("🚀 Testing quick prompt...")
quick_result = tester.test_prompt(QUICK_PROMPT, image_path)

print(f"\n✅ Completed in {quick_result['processing_time']:.2f}s")
print(f"📏 Response length: {len(quick_result['response'])} chars")
print("\n📄 Output:")
display(Markdown(quick_result['response']))

In [ ]:
# Test multiple prompts in sequence
PROMPT_VARIANTS = [
    "Convert to markdown.",
    "Extract as markdown with proper formatting.",
    "Create a markdown version of this document."
]

print("🔬 Testing prompt variants...\n")
for i, prompt in enumerate(PROMPT_VARIANTS, 1):
    print(f"Variant {i}: {prompt[:50]}...")
    result = tester.test_prompt(prompt, image_path)
    print(f"   Time: {result['processing_time']:.2f}s | Length: {len(result['response'])} chars\n")

## Export Results

In [ ]:
# Save the last result to a file
if model_response:
    output_dir = Path("../output")
    output_dir.mkdir(exist_ok=True)
    output_file = output_dir / f"experimental_output_{MODEL}_{int(time.time())}.md"
    output_file.write_text(model_response)
    print(f"💾 Saved output to: {output_file.relative_to('..')}")
    print(f"   File size: {len(model_response)} bytes")
else:
    print("❌ No output to save")

## Tips for Using This Notebook

1. **Edit the Configuration**: Modify the `EXPERIMENTAL_PROMPT` in the configuration cell
2. **Run All Cells**: Use Cell → Run All to execute the entire notebook
3. **Quick Iteration**: Use the "Quick Iteration Zone" cells for rapid testing
4. **Compare Models**: Change `MODEL` between "llama" and "internvl3" to compare
5. **Adjust Tokens**: Increase `MAX_TOKENS` if output is being cut off
6. **Save Results**: The last cell exports your results to a markdown file

### Keyboard Shortcuts
- `Shift + Enter`: Run cell and move to next
- `Ctrl + Enter`: Run cell and stay
- `Alt + Enter`: Run cell and insert new cell below